In [1]:
import os
import json
import re
import string
import time
import pandas as pd
from sklearn.model_selection import train_test_split

# =========================
# Config
# =========================
RAW_FILE = "../data/raw/all_emotions-2.csv"
SHARED_DATASET_FILE = "../data/processed/cleaned_emotions_shared.csv"
SHARED_SPLIT_FILE = "../data/splits/shared_split.json"

RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
NEUTRAL_LIMIT = 500

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/splits", exist_ok=True)

start_time = time.time()

# =========================
# Clean text
# =========================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

# =========================
# Load dataset
# =========================
df = pd.read_csv(RAW_FILE)
df.columns = df.columns.str.strip().str.lower()

required_cols = {"speaker", "utterance", "emotion"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"ไฟล์ต้องมีคอลัมน์ {required_cols}")

# =========================
# Filter patient only
# =========================
df["speaker"] = df["speaker"].astype(str).str.strip().str.lower()
df = df[df["speaker"] == "patient"].copy()

print("After filtering patient:", df.shape)

# =========================
# Keep needed columns
# =========================
df = df[["utterance", "emotion"]].dropna().copy()
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()
df["text"] = df["utterance"].apply(clean_text)
df = df[df["text"] != ""].copy()

# =========================
# Limit neutral
# =========================
neutral_df = df[df["emotion"] == "neutral"].copy()
other_df = df[df["emotion"] != "neutral"].copy()

if len(neutral_df) > NEUTRAL_LIMIT:
    neutral_df = neutral_df.sample(n=NEUTRAL_LIMIT, random_state=RANDOM_STATE)

df = pd.concat([neutral_df, other_df], axis=0)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# =========================
# Final dataset
# =========================
df = df[["text", "emotion"]].rename(columns={"emotion": "label"})
df["row_id"] = range(len(df))

print("\nFinal dataset shape:", df.shape)

label_counts = df["label"].value_counts().sort_index()
print("\nEmotion counts (all data):")
print(label_counts)

print("\nNumber of unique emotions:", df["label"].nunique())
print("Emotion labels:", sorted(df["label"].unique().tolist()))

df.to_csv(SHARED_DATASET_FILE, index=False, encoding="utf-8-sig")

# =========================
# Shared split
# =========================
train_df, temp_df = train_test_split(
    df,
    test_size=(TEST_SIZE + VAL_SIZE),
    stratify=df["label"],
    random_state=RANDOM_STATE
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_SIZE / (TEST_SIZE + VAL_SIZE),
    stratify=temp_df["label"],
    random_state=RANDOM_STATE
)

split_dict = {
    "train_ids": train_df["row_id"].tolist(),
    "val_ids": val_df["row_id"].tolist(),
    "test_ids": test_df["row_id"].tolist()
}

with open(SHARED_SPLIT_FILE, "w", encoding="utf-8") as f:
    json.dump(split_dict, f, ensure_ascii=False, indent=2)

print("\nSaved:")
print("-", SHARED_DATASET_FILE)
print("-", SHARED_SPLIT_FILE)

print("\nSplit sizes:")
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

print("\nTrain label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nVal label distribution:")
print(val_df["label"].value_counts().sort_index())

print("\nTest label distribution:")
print(test_df["label"].value_counts().sort_index())

elapsed = time.time() - start_time
print(f"\nPreprocessing time: {elapsed:.2f} seconds")
print("\nDone preprocessing")

After filtering patient: (4516, 3)

Final dataset shape: (2160, 3)

Emotion counts (all data):
label
angry      326
disgust    427
fear       395
happy      234
neutral    500
sad        278
Name: count, dtype: int64

Number of unique emotions: 6
Emotion labels: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']

Saved:
- ../data/processed/cleaned_emotions_shared.csv
- ../data/splits/shared_split.json

Split sizes:
Train: 1512
Val  : 324
Test : 324

Train label distribution:
label
angry      228
disgust    299
fear       276
happy      164
neutral    350
sad        195
Name: count, dtype: int64

Val label distribution:
label
angry      49
disgust    64
fear       59
happy      35
neutral    75
sad        42
Name: count, dtype: int64

Test label distribution:
label
angry      49
disgust    64
fear       60
happy      35
neutral    75
sad        41
Name: count, dtype: int64

Preprocessing time: 0.17 seconds

Done preprocessing
